# Input formats

In [1]:
import earthkit.data as ekd

## Computing the wind speed

First, we generate the input data for array, Xarray and fieldlist.

In [2]:
from earthkit.meteo import wind

# get Xarray dataset
ds = ekd.from_source("sample", "era5_tquv_pl_subarea.nc").to_xarray()

# get GRIB fieldlist
fl = ekd.from_source("sample", "era5_tquv_pl_subarea.grib").to_fieldlist()

# get input numpy arrays
u_arr = ds.u.values
v_arr = ds.v.values

era5_tquv_pl_subarea.nc:   0%|          | 0.00/14.6M [00:00<?, ?B/s]

era5_tquv_pl_subarea.grib:   0%|          | 0.00/16.6M [00:00<?, ?B/s]

### Using the high level interface

In [3]:
# import the high level wind module

In [4]:
# compute with numpy array
sp_arr = wind.speed(u_arr, v_arr)
type(sp_arr), sp_arr.shape

(numpy.ndarray, (4, 3, 301, 601))

In [5]:
# compute with Xarray, the inputs are dataarrays
sp_xr = wind.speed(ds.u, ds.v)

# set the metadata manually (this will be automatic in the future)
sp_xr.attrs["standard_name"] = "wind_speed"
sp_xr.attrs["long_name"] = "wind_speed"
sp_xr = sp_xr.rename("ws")
sp_xr

<xarray.DataArray 'ws' (valid_time: 4, pressure_level: 3, latitude: 301,
                        longitude: 601)> Size: 9MB
array([[[[7.21209857e-04, 7.21209857e-04, 7.21209857e-04, ...,
          7.21209857e-04, 7.21209857e-04, 7.21209857e-04],
         [9.15587044e+00, 9.15572071e+00, 9.15571594e+00, ...,
          9.67847347e+00, 9.67668724e+00, 9.67551231e+00],
         [8.77466106e+00, 8.77321625e+00, 8.77142429e+00, ...,
          9.91900063e+00, 9.91710377e+00, 9.91543484e+00],
         ...,
         [1.12219691e+00, 1.00199103e+00, 8.06946635e-01, ...,
          9.93535876e-01, 4.55393404e-01, 6.65705085e-01],
         [4.73718941e-01, 4.93476927e-01, 5.56098044e-01, ...,
          5.67884982e-01, 7.18522072e-01, 4.60717976e-01],
         [1.05491556e-01, 3.18013817e-01, 4.84191030e-01, ...,
          9.93801832e-01, 6.87026620e-01, 4.87377763e-01]],

        [[1.04129361e-03, 1.04129361e-03, 1.04129361e-03, ...,
          1.04129361e-03, 1.04129361e-03, 1.04129361e-03],
         [8.28795815e+00, 8.29496670e+00, 8.30037975e+00, ...,
          8.25793839e+00, 8.25712204e+00, 8.25645351e+00],
         [8.83082581e+00, 8.84640408e+00, 8.86025906e+00, ...,
          8.32423401e+00, 8.32343578e+00, 8.32278347e+00],
...
          1.70103025e+00, 1.26007354e+00, 1.77098072e+00],
         [2.65315723e+00, 2.73314214e+00, 2.64034247e+00, ...,
          2.76825452e+00, 2.20397377e+00, 2.17171168e+00],
         [2.73072147e+00, 2.88751960e+00, 2.87780690e+00, ...,
          2.61288738e+00, 2.68276882e+00, 2.50446677e+00]],

        [[6.33009884e-04, 6.33009884e-04, 6.33009884e-04, ...,
          6.33009884e-04, 6.33009884e-04, 6.33009884e-04],
         [7.87639523e+00, 7.87548780e+00, 7.87472439e+00, ...,
          7.88470078e+00, 7.88728619e+00, 7.89145756e+00],
         [7.69088745e+00, 7.68987322e+00, 7.68869591e+00, ...,
          7.31118202e+00, 7.31721783e+00, 7.32209921e+00],
         ...,
         [3.76584339e+00, 4.03104496e+00, 4.31973362e+00, ...,
          9.29071903e+00, 7.57689333e+00, 6.01164341e+00],
         [4.17330790e+00, 4.29592800e+00, 4.42056227e+00, ...,
          8.29728794e+00, 6.92111635e+00, 5.99423361e+00],
         [4.42543697e+00, 4.44226933e+00, 4.52274704e+00, ...,
          6.44288826e+00, 6.02955246e+00, 6.10808563e+00]]]],
      shape=(4, 3, 301, 601), dtype=float32)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 32B 2016-09-26 ... 2016-09-26...
  * pressure_level  (pressure_level) float64 24B 925.0 850.0 700.0
  * latitude        (latitude) float64 2kB 90.0 89.75 89.5 ... 15.5 15.25 15.0
  * longitude       (longitude) float64 5kB -100.0 -99.75 -99.5 ... 49.75 50.0
Attributes: (12/31)
    GRIB_paramId:                             131
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      180901
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           u
    GRIB_totalNumber:                         0
    GRIB_units:                               m s**-1
    long_name:                                wind_speed
    units:                                    m s**-1
    standard_name:                            wind_speed

In [6]:
# compute with fieldlist

# the inputs are fieldlists of u and v components.
# We must ensure the u and v fields are properly paired up. Our data is already sorted
# so we do not need to deal with it now.
u_fl = fl.sel({"parameter.variable": "u"})
v_fl = fl.sel({"parameter.variable": "v"})

# compute
sp_fl = wind.speed(u_fl, v_fl)
sp_fl.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,ws,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,700,pressure,0,regular_ll
1,ws,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,850,pressure,0,regular_ll
2,ws,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,925,pressure,0,regular_ll
3,ws,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,700,pressure,0,regular_ll
4,ws,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,850,pressure,0,regular_ll
5,ws,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,925,pressure,0,regular_ll
6,ws,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,700,pressure,0,regular_ll
7,ws,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,850,pressure,0,regular_ll
8,ws,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,925,pressure,0,regular_ll
9,ws,2016-09-26 18:00:00,2016-09-26 18:00:00,0 days,700,pressure,0,regular_ll


### Using the format specific submodules

We can perform the computations directly by calling the method from the format specific submodule. With this we can avoid the dispatching and the unnecessary checks it involves.

In [7]:
# import the array wind submodule
import earthkit.meteo.wind.array as wind_array

# compute with numpy array
sp_arr = wind_array.speed(u_arr, v_arr)
type(sp_arr), sp_arr.shape

(numpy.ndarray, (4, 3, 301, 601))

In [8]:
# import the Xarray wind submodule
import earthkit.meteo.wind.xarray as wind_xr

# compute with Xarray, the inputs are dataarrays
sp_xr = wind_xr.speed(ds.u, ds.v)

# set the metadata manually (this will be automatic in the future)
sp_xr.attrs["standard_name"] = "wind_speed"
sp_xr.attrs["long_name"] = "wind_speed"
sp_xr = sp_xr.rename("ws")
type(sp_xr)

xarray.core.dataarray.DataArray

In [9]:
# import the fieldlist wind module
import earthkit.meteo.wind.fieldlist as wind_fl

# extract u and v fieldlists
u_fl = fl.sel({"parameter.variable": "u"})
v_fl = fl.sel({"parameter.variable": "v"})

# compute
sp_fl = wind_fl.speed(u_fl, v_fl)
type(sp_fl)

earthkit.data.indexing.simple.SimpleFieldList